# Antes del laboratorio 3.3: PyTorch, Gymnasium y REINFORCE

**Cuaderno de preparación para el laboratorio «REINFORCE desde cero en Snake»**  
**Nivel:** primera experiencia con PyTorch · **Modalidad:** explicaciones, ejemplos ejecutables y prácticas autocorregibles  
**Tiempo orientativo de trabajo del alumnado:** 3–4 horas, repartibles en dos sesiones.

> **Propósito.** Entender las herramientas y su conexión con el aprendizaje por refuerzo antes de completar `lab_3_3_reinforce.py`. Este cuaderno **no es la solución de la entrega**: trabaja los conceptos con tensores pequeños, una red de ejemplo y `CartPole-v1`, y deja los `TODO` del Snake al alumnado.

### Al terminar serás capaz de

1. Crear tensores de PyTorch y razonar sobre su `shape`, `dtype` y gradientes.
2. Definir una MLP con `torch.nn`, convertir **logits** en una distribución de acciones y muestrear acciones.
3. Comprender `backward()`, `detach()` y `Adam`.
4. Calcular retornos descontados, interpretar la pérdida de REINFORCE y usar un baseline.
5. Interactuar con Gymnasium, registrar una trayectoria, depurar errores y analizar resultados.
6. Leer con criterio la interfaz de los siete `TODO` del laboratorio.

**Convención:** `[B, D]` significa *B elementos en un lote, D características por elemento*; `[T, D]` significa *T pasos temporales*.

## 0. Preparación del entorno

El ejercicio utiliza **Python 3**, **NumPy**, **PyTorch** y **Gymnasium**. Para supervisar experimentos se usa **TensorBoard**; para gráficas, **Matplotlib**. El script suministrado utiliza otros archivos del repositorio (`common3.py`, `snake_design.py` y `plot_results.py`) que **no se incluyen** en este cuaderno.

**Entorno recomendado:** ejecutar en Jupyter/VS Code o Google Colab. Basta **CPU** para seguir los ejemplos; una GPU no es requisito. Usa el entorno virtual del repositorio si el profesorado ha proporcionado uno.

La siguiente celda instala únicamente las bibliotecas que falten. Si el centro ya proporciona un entorno configurado, se puede omitir.

In [1]:
# Instalación asistida: se ejecuta únicamente si faltan paquetes.
import importlib.util
import subprocess
import sys

paquetes = {
    "torch": "torch",
    "numpy": "numpy",
    "gymnasium": "gymnasium",
    "matplotlib": "matplotlib",
    "tensorboard": "tensorboard",
    "imageio": "imageio",
}
faltantes = [pip_nombre for modulo, pip_nombre in paquetes.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
else:
    print("Dependencias principales disponibles.")

Instalando: torch, numpy, gymnasium, matplotlib, tensorboard, imageio


In [2]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
from torch.distributions import Categorical

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)
print("Dispositivo usado:", "cpu")
torch.manual_seed(7)
np.random.seed(7)

Python: 3.11.1
PyTorch: 2.15.0+cpu
Gymnasium: 1.4.0
Dispositivo usado: cpu


## 1. Del problema de Snake a un agente que aprende

En cada instante, el agente recibe una **observación** $s_t$, elige una **acción** $a_t$, obtiene una **recompensa** $r_{t+1}$ y pasa al siguiente estado observable. Una secuencia completa hasta el final de la partida es un **episodio**.

| Elemento | Qué significa en la entrega |
|---|---|
| Entorno | Snake, construido en `make_snake_env(...)` |
| Observación | Vector numérico que describe la situación de la serpiente |
| Acciones | Acciones discretas disponibles (`env.action_space.n`) |
| Política $\pi_\theta(a\mid s)$ | Red neuronal que decide **probabilísticamente** qué acción ejecutar |
| Retorno $G_t$ | Suma de recompensas futuras descontadas desde el paso $t$ |
| Función de valor $\hat v_w(s)$ | Red que estima el retorno esperado a partir de una observación |
| Evaluación | Recompensa acumulada y, en Snake, **manzanas por partida** (`score`) |

**Objetivo de REINFORCE:** modificar los parámetros $\theta$ para aumentar la probabilidad de las acciones que se asociaron a buenos resultados. Se entrena a partir de **episodios completos**.

No confundas dos procesos: **entrenar** suele requerir *muestrear* acciones para explorar; **evaluar**, en el código de la entrega, utiliza la acción más probable (`argmax`).

## 2. PyTorch: tensores, formas y tipos

Un `torch.Tensor` es una estructura numérica similar a un `ndarray` de NumPy, pero compatible con diferenciación automática y ejecución acelerada. Tres preguntas deben acompañar a cada tensor: **¿qué contiene?, ¿qué forma tiene?, ¿qué tipo utiliza?**

- `float32`: observaciones, logits, retornos, recompensas y pérdidas.
- `int64` (`torch.long`): índices de acciones discretas.
- `shape`: PyTorch no adivina qué eje representa el lote; hay que diseñarlo.

In [3]:
obs_numpy = np.array([0.2, -0.1, 0.5, 1.0], dtype=np.float32)
obs = torch.as_tensor(obs_numpy, dtype=torch.float32)
acciones = torch.tensor([0, 1, 0], dtype=torch.long)
recompensas = torch.tensor([1.0, 0.0, 2.0], dtype=torch.float32)

print("Observación:", obs, "| shape:", tuple(obs.shape), "| dtype:", obs.dtype)
print("Acciones:", acciones, "| shape:", tuple(acciones.shape), "| dtype:", acciones.dtype)
print("Recompensas:", recompensas)

assert obs.shape == (4,) and obs.dtype == torch.float32
assert acciones.shape == (3,) and acciones.dtype == torch.int64

Observación: tensor([ 0.2000, -0.1000,  0.5000,  1.0000]) | shape: (4,) | dtype: torch.float32
Acciones: tensor([0, 1, 0]) | shape: (3,) | dtype: torch.int64
Recompensas: tensor([1., 0., 2.])


In [4]:
# Un solo estado frente a un lote de estados.
estado_unico = torch.randn(4)       # [D]
lote_estados = torch.randn(5, 4)    # [B, D]
print("Un estado:", estado_unico.shape)
print("Cinco estados:", lote_estados.shape)

# torch.stack crea un eje nuevo; torch.cat concatena sobre uno existente.
pasos = [torch.randn(4) for _ in range(3)]
trayectoria = torch.stack(pasos)      # [T=3, D=4]
print("Trayectoria:", trayectoria.shape)
assert trayectoria.shape == (3, 4)

Un estado: torch.Size([4])
Cinco estados: torch.Size([5, 4])
Trayectoria: torch.Size([3, 4])


In [5]:
# Demostración útil: squeeze(-1) elimina SOLO el último eje si su tamaño es uno.
valores_columna = torch.randn(6, 1)
valores_vector = valores_columna.squeeze(-1)
print(valores_columna.shape, "->", valores_vector.shape)
assert valores_vector.shape == (6,)

# Cuidado: squeeze() sin dimensión también podría eliminar el eje B si B=1.
assert torch.randn(1, 1).squeeze(-1).shape == (1,)

torch.Size([6, 1]) -> torch.Size([6])


### Práctica 1 · Formas y tipos

Completa la celda para crear un tensor `x_ej` con **4 observaciones y 6 características por observación**, y otro `a_ej` con cuatro índices de acción válidos para un problema de tres acciones. Sustituye los `None`. La comprobación se activará al completar ambas variables.

In [ ]:
x_ej = torch.randn(4, 6)  # TODO estudiante: forma [4, 6], float32
a_ej = torch.randn(4)  # TODO estudiante: forma [4], long, valores entre 0 y 2

if x_ej is None or a_ej is None:
    print("Práctica 1 pendiente: define x_ej y a_ej.")
else:
    assert x_ej.shape == (4, 6) and x_ej.dtype == torch.float32
    assert a_ej.shape == (4,) and a_ej.dtype == torch.long
    assert torch.all((a_ej >= 0) & (a_ej < 3))
    print("Práctica 1 superada.")

## 3. Redes neuronales con `torch.nn`

Una **MLP** transforma un vector de entrada mediante capas `Linear` y activaciones no lineales. En la entrega se pide una política con **dos capas ocultas `tanh`** y una capa de salida lineal. Aquí usaremos otra configuración para practicar la sintaxis sin entregar `PolicyNet` implementada.

La última capa de una política de acciones discretas genera un valor real por acción: esos valores son **logits**. **No añadiremos `Softmax`** porque `Categorical(logits=...)` lo aplica internamente de forma adecuada.

In [ ]:
# Red de EJEMPLO, distinta de la arquitectura solicitada en Snake.
class RedDidactica(nn.Module):
    def __init__(self, entradas: int, salidas: int):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(entradas, 12),
            nn.Tanh(),
            nn.Linear(12, salidas),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.capas(x)

red = RedDidactica(entradas=4, salidas=2)
print(red)
print("Entrada individual:", red(torch.zeros(4)).shape)
print("Entrada en lote:", red(torch.zeros(7, 4)).shape)
assert red(torch.zeros(4)).shape == (2,)
assert red(torch.zeros(7, 4)).shape == (7, 2)
print("Parámetros entrenables:", sum(p.numel() for p in red.parameters()))

**Lectura de la arquitectura:** `Linear(4,12)` espera cuatro características y genera doce; `Tanh()` introduce no linealidad; `Linear(12,2)` produce dos logits. `nn.Module` registra los parámetros automáticamente y `forward` define el cálculo.

**Errores habituales:** crear capas dentro de `forward` (recrea pesos continuamente), aplicar `softmax` y luego usar `Categorical(logits=...)` (doble transformación), o forzar a mano un eje de lote que no corresponde.

### Práctica 2 · Diseña una MLP

Escribe en el laboratorio, **no aquí**, la arquitectura solicitada por `TODO 1`: entrada `obs_dim`, capas ocultas de tamaño `hidden`, activaciones `tanh` y salida `n_actions`. Antes, dibuja en papel sus dimensiones para una entrada `[7, obs_dim]` y para `[obs_dim]`.

## 4. De logits a probabilidades, acciones y `log_prob`

Para un conjunto de acciones discretas:

$$\pi_\theta(a=i\mid s) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Aquí $z_i$ es el **logit** de la acción $i$. La clase `torch.distributions.Categorical` permite:

- `sample()`: elegir una acción de manera aleatoria según las probabilidades.
- `log_prob(accion)`: obtener $\log \pi_\theta(a\mid s)$, que se usa en REINFORCE.
- `probs`: inspeccionar probabilidades; `entropy()` sirve para diagnosticar exploración.

In [ ]:
logits = torch.tensor([0.0, 1.0, 2.0])
d = Categorical(logits=logits)
a = d.sample()
log_p = d.log_prob(a)
print("Logits:", logits.tolist())
print("Probabilidades:", d.probs.tolist())
print("Acción muestreada:", a.item(), "| log-probabilidad:", log_p.item())
print("Más probable:", torch.argmax(logits).item())
assert torch.isclose(d.probs.sum(), torch.tensor(1.0))
assert a.ndim == 0 and log_p.ndim == 0

In [ ]:
# Si hay varios estados, cada fila crea UNA distribución categórica.
logits_lote = torch.tensor([[2.0, 0.0], [0.2, -0.3], [-2.0, 3.0]])
d_lote = Categorical(logits=logits_lote)
a_lote = d_lote.sample()                # [B]
lp_lote = d_lote.log_prob(a_lote)       # [B]
print("logits", logits_lote.shape, "acciones", a_lote.shape, "log_probs", lp_lote.shape)
assert a_lote.shape == (3,) and lp_lote.shape == (3,)

### Práctica 3 · Interpretar la distribución

Con logits `[0, 0, 0, 0]`, ¿qué probabilidad tiene cada acción? Compruébalo creando tu propia distribución. Explica por qué **muestrear** y elegir `argmax` no es lo mismo, incluso con logits `[1, 0, 0, 0]`.

In [ ]:
probabilidades_ej = None  # TODO estudiante: tensor de probabilidades para 4 logits nulos

if probabilidades_ej is None:
    print("Práctica 3 pendiente.")
else:
    assert torch.allclose(probabilidades_ej, torch.full((4,), 0.25))
    print("Práctica 3 superada.")

## 5. Autograd: por qué los `log_probs` deben conservar el gradiente

PyTorch registra operaciones que dependen de parámetros entrenables. Si al final calculas una **pérdida escalar**, `loss.backward()` calcula las derivadas de esa pérdida respecto a los parámetros. El optimizador utiliza los gradientes para actualizar la red.

**Patrón básico:** `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
perdida_simple = (w - 5.0) ** 2
perdida_simple.backward()
print("Pérdida:", perdida_simple.item(), "| Gradiente en w:", w.grad.item())
assert torch.isclose(w.grad, torch.tensor(-6.0))

In [ ]:
# Demostración del paso de optimización sobre una red pequeñita.
torch.manual_seed(10)
modelo_demo = nn.Linear(2, 1)
optim_demo = torch.optim.SGD(modelo_demo.parameters(), lr=0.05)
entradas_demo = torch.tensor([[1.0, 2.0], [2.0, 1.0], [1.0, 1.0]])
objetivo_demo = torch.tensor([[3.0], [3.0], [2.0]])

pesos_antes = modelo_demo.weight.detach().clone()
prevision = modelo_demo(entradas_demo)
loss_demo = ((prevision - objetivo_demo) ** 2).mean()
optim_demo.zero_grad()
loss_demo.backward()
print("Norma del gradiente:", modelo_demo.weight.grad.norm().item())
optim_demo.step()
print("¿Han cambiado los pesos?", not torch.allclose(pesos_antes, modelo_demo.weight.detach()))
assert not torch.allclose(pesos_antes, modelo_demo.weight.detach())

### `detach()`, `item()` y `no_grad()` no son equivalentes

| Expresión | Uso correcto | Qué ocurre con autograd |
|---|---|---|
| `tensor.detach()` | Constante para una parte de la pérdida; observación/registro | Separa el tensor de su historial de gradientes |
| `tensor.item()` | Mostrar un escalar Python | Deja de ser tensor: no se puede propagar gradiente |
| `with torch.no_grad(): ...` | Evaluación sin actualización | No construye el grafo de derivadas |
| `red.eval()` | Cambia el modo de capas como dropout/batchnorm | **No desactiva por sí solo** los gradientes |

**Regla central de `TODO 3`:** hay que guardar `log_probs` **como tensores con gradiente**, sin llamar a `item()` ni `detach()` al almacenarlos. Lo que se convierte a `int` para el entorno es **la acción**, no su log-probabilidad.

In [ ]:
# Verificación: los log_probs provenientes de la red conservan el grafo.
estado_grad = torch.tensor([0.1, -0.2, 0.3, 0.4])
dist_grad = Categorical(logits=red(estado_grad))
accion_grad = dist_grad.sample()
log_prob_grad = dist_grad.log_prob(accion_grad)
print("log_prob.requires_grad:", log_prob_grad.requires_grad)
print("log_prob.detach().requires_grad:", log_prob_grad.detach().requires_grad)
assert log_prob_grad.requires_grad and not log_prob_grad.detach().requires_grad

## 6. Retornos descontados: el cálculo que alimenta REINFORCE

La recompensa inmediata $r_{t+1}$ no es el retorno. Desde cada paso se suman recompensas futuras descontadas:

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots$$

$\gamma \in [0,1]$ regula la importancia del futuro. El cálculo eficiente utiliza la **recurrencia hacia atrás**:

$$G_t=r_{t+1}+\gamma G_{t+1}, \qquad G_T=0.$$

Ejemplo para recompensas `[1, 1, 1]` y $\gamma=0.5$:

| Paso | Cálculo | Retorno |
|---|---|---:|
| $t=2$ | $1$ | $1.00$ |
| $t=1$ | $1 + 0.5 \cdot 1$ | $1.50$ |
| $t=0$ | $1 + 0.5 \cdot 1.5$ | $1.75$ |

En el `TODO 2` tienes que **programar** esta idea para una lista de longitud arbitraria y devolver un `torch.Tensor` de tipo `float32`. El ejemplo siguiente verifica los cálculos **sin implementar** la función de la entrega.

In [ ]:
# Cálculo explícito para un caso de tres recompensas.
rewards_demo = [1.0, 1.0, 1.0]
gamma_demo = 0.5
ultimo = rewards_demo[2]
medio = rewards_demo[1] + gamma_demo * ultimo
primero = rewards_demo[0] + gamma_demo * medio
retornos_demo = torch.tensor([primero, medio, ultimo], dtype=torch.float32)
print("Retornos:", retornos_demo)
assert torch.allclose(retornos_demo, torch.tensor([1.75, 1.50, 1.0]))

### Práctica 4 · Cálculo y razonamiento

Sin copiar el `TODO 2`, diseña el pseudocódigo para resolver una lista de $T$ recompensas recorriéndola **una vez de derecha a izquierda**. Luego comprueba manualmente el resultado de `[0, 0, 10]` cuando $\gamma=0.9$. ¿Qué valor debería ocupar `G[0]`?

In [ ]:
g0_practica = None  # TODO estudiante: calcula G[0] como número Python
if g0_practica is None:
    print("Práctica 4 pendiente.")
else:
    assert abs(g0_practica - 8.1) < 1e-6
    print("Práctica 4 superada.")

## 7. Gymnasium: crear un entorno, reiniciar y dar un paso

Mientras no hayas completado `snake_design.py`, **CartPole-v1** es el entorno más cómodo para practicar la interfaz.

```python
obs, info = env.reset(seed=0)
obs_siguiente, recompensa, terminated, truncated, info = env.step(accion)
done = terminated or truncated
```

- `terminated`: estado terminal según la dinámica/objetivo del problema.
- `truncated`: episodio interrumpido por una condición externa al criterio terminal, como un límite de pasos.
- En ambos casos **debe terminar el bucle de recolección**. La distinción sí importa si más adelante utilizas algoritmos con *bootstrap* de valores.
- El `info` del Snake puede incluir la métrica `score`; en CartPole no debes suponer que aparece.

In [ ]:
env = gym.make("CartPole-v1")
obs_cp, info_cp = env.reset(seed=4)
env.action_space.seed(4)
print("Espacio de observaciones:", env.observation_space)
print("Espacio de acciones:", env.action_space)
print("Observación inicial:", obs_cp, "shape:", obs_cp.shape)
accion_aleatoria = env.action_space.sample()
nueva_obs, reward, terminated, truncated, info = env.step(accion_aleatoria)
print("Acción:", accion_aleatoria, "| recompensa:", reward)
print("terminated:", terminated, "| truncated:", truncated)
assert obs_cp.shape == (4,) and env.action_space.n == 2

### Jugar un episodio con una política **aleatoria**

Aquí veremos el ciclo completo sin entrenar ni usar una red neuronal. La colección de la entrega es más exigente: debe **muestrear desde la política de PyTorch**, registrar las observaciones **anteriores** a cada acción y guardar los `log_probs` con gradiente.

In [ ]:
obs_cp, _ = env.reset(seed=12)
terminado = False
recompensas_cp = []
acciones_cp = []
observaciones_cp = []

while not terminado:
    observaciones_cp.append(np.asarray(obs_cp, dtype=np.float32).copy())
    action = env.action_space.sample()
    acciones_cp.append(action)
    obs_cp, r, term, trunc, info = env.step(action)
    recompensas_cp.append(float(r))
    terminado = term or trunc

print("Pasos:", len(recompensas_cp))
print("Retorno sin descuento:", sum(recompensas_cp))
print("Observaciones guardadas:", np.asarray(observaciones_cp).shape)
assert len(observaciones_cp) == len(acciones_cp) == len(recompensas_cp)
env.close()

## 8. Cómo se representa un episodio en la entrega

`collect_episode` tiene que devolver un **diccionario** con estas claves y contratos:

| Clave | Tipo y forma | Significado |
|---|---|---|
| `obs` | `float32` `[T, obs_dim]` | Estados en los que se eligieron acciones |
| `actions` | `long` `[T]` | Índices de acción |
| `log_probs` | `float32` `[T]` **con gradiente** | Log-probabilidades de las acciones elegidas |
| `rewards` | `list[float]` con longitud `T` | Recompensas inmediatas |
| `return` | `float` | Suma no descontada de recompensas |
| `length` | `int` | Duración `T` del episodio |
| `score` | `int` | Manzanas, o cero si no aparece en `info` |

**Patrón importante:** `torch.stack(lista_tensores)` une salidas por paso; si almacenas un resultado de PyTorch usando `float(tensor)` o `tensor.item()`, pierdes el grafo de gradientes.

**No es suficiente** almacenar logits y usar una acción generada al azar desde `env.action_space`: se debe usar la distribución de la **política actual**. Una forma de comprobar coherencia es volver a calcular `distribution(obs).log_prob(actions)` y verificar que coincide con los valores guardados (desconectados únicamente para comparar).

In [ ]:
# Ejemplo AISLADO de formas de un episodio (datos sintéticos, no una solución).
T, D = 5, 4
obs_sinteticas = torch.randn(T, D)
acciones_sinteticas = torch.tensor([0, 1, 1, 0, 1], dtype=torch.long)
dist_sintetica = Categorical(logits=red(obs_sinteticas))
log_probs_sinteticas = dist_sintetica.log_prob(acciones_sinteticas)

print("obs:", obs_sinteticas.shape, "dtype:", obs_sinteticas.dtype)
print("actions:", acciones_sinteticas.shape, "dtype:", acciones_sinteticas.dtype)
print("log_probs:", log_probs_sinteticas.shape,
      "requires_grad:", log_probs_sinteticas.requires_grad)
assert log_probs_sinteticas.shape == (T,)
assert log_probs_sinteticas.requires_grad

### Práctica 5 · Auditoría de contratos

Piensa qué fallaría si: (a) `obs` tiene forma `[obs_dim, T]`, (b) `actions` es `float32`, (c) `log_probs` se construye con `torch.tensor(lista_log_probs)`, o (d) se almacena la **observación siguiente** en vez de aquella con la que se eligió la acción. Para cada caso indica **qué comprobación de `--check`** podría descubrirlo.

## 9. La pérdida de REINFORCE: actualizar la probabilidad de las acciones

El gradiente de política se estima a partir de episodios muestreados:

$$\nabla_\theta J(\theta) \approx \frac{1}{N}\sum_{t=1}^{N} G_t\,\nabla_\theta\log\pi_\theta(a_t\mid s_t).$$

Como los optimizadores de PyTorch **minimizan** pérdidas, se utiliza el negativo:

$$\mathcal L_{\pi}=-\frac{1}{N}\sum_{t=1}^{N}G_t\log\pi_\theta(a_t\mid s_t).$$

En el laboratorio, $N$ es el **número total de pasos de todos los episodios del lote**, no el número de episodios. Es importante entender el signo: si un retorno es positivo, interesa aumentar la log-probabilidad de la acción que lo produjo. Si el retorno relativo es negativo, se desincentiva.

Vamos a demostrar el **cálculo en un pequeño lote sintético**. No implementaremos `policy_loss(episodes, ...)`.

In [ ]:
torch.manual_seed(21)
red_actualizable = RedDidactica(entradas=4, salidas=2)
optim_pi = torch.optim.Adam(red_actualizable.parameters(), lr=1e-3)
mini_obs = torch.randn(4, 4)
mini_actions = torch.tensor([0, 1, 0, 1], dtype=torch.long)
mini_G = torch.tensor([3.0, 2.0, 1.0, 0.5], dtype=torch.float32)

log_probs_mini = Categorical(logits=red_actualizable(mini_obs)).log_prob(mini_actions)
loss_pi_demo = -(log_probs_mini * mini_G).mean()
optim_pi.zero_grad()
loss_pi_demo.backward()
print("Pérdida escalar:", loss_pi_demo.item())
print("¿Tiene gradiente un peso de la política?", red_actualizable.capas[0].weight.grad is not None)
optim_pi.step()
assert loss_pi_demo.ndim == 0

### Práctica 6 · El signo de la pérdida

Imagina un lote con recompensas futuras positivas y analiza qué hace `loss = +(log_probs * G).mean()` al minimizarlo. ¿Aumenta o reduce la probabilidad de las acciones? Explica qué signo debe llevar la pérdida de `TODO 4` y por qué.

## 10. Normalización: estabilidad y precauciones

El enunciado permite normalizar los retornos restando la media del lote y dividiendo por su desviación típica más $10^{-8}$:

$$\tilde G_t = \frac{G_t-\bar G}{\mathrm{std}(G)+10^{-8}}.$$

Esto modifica la escala de la señal de aprendizaje y puede facilitar el entrenamiento; también altera la interpretación de los pesos, por lo que conviene **explicitar qué normalizas y cómo** en el informe.

**Importante:** la normalización es sobre **los pasos de todo el lote**. Con un único valor, algunas implementaciones de desviación estándar muestral producen `nan`; el ejemplo usa `unbiased=False` para mostrar una alternativa estable. Si todos los retornos son iguales, el vector centrado será nulo y la actualización puede desaparecer.

In [ ]:
G_lote = torch.tensor([1.0, 2.0, 5.0, 6.0], dtype=torch.float32)
G_norm = (G_lote - G_lote.mean()) / (G_lote.std(unbiased=False) + 1e-8)
print("Retornos originales:", G_lote.tolist())
print("Retornos normalizados:", G_norm.tolist())
print("Media normalizada:", G_norm.mean().item())
assert abs(G_norm.mean().item()) < 1e-6

constantes = torch.ones(3)
constantes_normalizadas = (constantes - constantes.mean()) / (constantes.std(unbiased=False) + 1e-8)
assert torch.isfinite(constantes_normalizadas).all()

## 11. Baseline: una red que predice el valor

REINFORCE *vanilla* pondera las log-probabilidades por $G_t$. Un **baseline** que depende del estado permite comparar el resultado observado con lo que se esperaba de ese estado:

$$\hat A_t = G_t - \hat v_w(s_t).$$

La pérdida de la política utiliza $\hat A_t$, mientras que la red de valor ajusta su predicción mediante error cuadrático medio:

$$\mathcal L_v = \frac{1}{N}\sum_t\big(\hat v_w(s_t)-G_t\big)^2.$$

**Dos redes, dos objetivos, dos optimizadores** (según el esqueleto de la entrega). El baseline **no debe recibir gradiente de `loss_pi`**. Para ello, una forma sencilla consiste en utilizar `value_predictions.detach()` al construir las ventajas del actor; `loss_v` sí usa las predicciones originales con gradiente.

**Precisión conceptual:** un baseline dependiente únicamente del estado no altera el gradiente de política esperado si se satisfacen las hipótesis del estimador; puede reducir la varianza, aunque una aproximación imperfecta y el uso de lotes finitos influyen en el resultado.

In [ ]:
# Demostración con una red distinta de la ValueNet pedida.
red_valor_demo = nn.Sequential(
    nn.Linear(4, 8),
    nn.Tanh(),
    nn.Linear(8, 1),
)
minilote = torch.randn(5, 4)
valores_predichos = red_valor_demo(minilote).squeeze(-1)    # [5]
objetivos_G = torch.tensor([1.0, 1.5, 0.0, 2.0, 1.0])
ventajas = objetivos_G - valores_predichos.detach()
loss_valor_demo = ((valores_predichos - objetivos_G) ** 2).mean()
print("Valores predichos:", valores_predichos.shape)
print("Ventajas sin gradiente:", ventajas.requires_grad)
print("Pérdida de valor:", loss_valor_demo.item())
assert valores_predichos.shape == (5,)
assert ventajas.requires_grad is False
assert loss_valor_demo.requires_grad

In [ ]:
# Verificamos separación de gradientes en un ejemplo independiente.
red_valor_demo.zero_grad()
dist_actor = Categorical(logits=red_actualizable(minilote))
a_actor = torch.tensor([0, 1, 0, 1, 1], dtype=torch.long)
loss_actor_aislada = -(dist_actor.log_prob(a_actor) * ventajas).mean()
loss_actor_aislada.backward()
assert all(p.grad is None for p in red_valor_demo.parameters())
print("Correcto: loss_actor no ha creado gradientes en la red de valor.")

red_valor_demo.zero_grad()
loss_valor_demo.backward()
assert any(p.grad is not None for p in red_valor_demo.parameters())
print("Correcto: loss_valor sí actualiza las derivadas de la red de valor.")

### Práctica 7 · Diseñar el `TODO 5`

1. ¿Qué dimensión debe tener la salida de `ValueNet` al recibir `[12, obs_dim]`? ¿Y `[1, obs_dim]`?
2. ¿Por qué `.squeeze(-1)` es preferible aquí a `.squeeze()`?
3. ¿Qué error aparece si el baseline se resta **sin** separar su gradiente cuando calculas `loss_pi.backward()`?
4. ¿Normalizarías `G`, la ventaja $\hat A$ o ninguna de ellas? Formula una hipótesis contrastable para tu informe.

## 12. El bucle de entrenamiento, sin entrenar todavía el Snake

El fichero de entrega ya contiene este **orden lógico**:

1. Crear entorno, política, optimizador y, si procede, red de valor.
2. Recopilar `batch_episodes` episodios usando **la misma versión de la política**.
3. Construir retornos, pérdida de política y pérdida de valor (opcional).
4. Ejecutar `zero_grad()`, `backward()`, `step()` para política; después, si procede, para valor.
5. Registrar indicadores; evaluar periódicamente con una política **determinista**.
6. Continuar hasta superar el presupuesto `--steps` y guardar resultados.

**Por qué importa el orden:** no es correcto modificar los pesos de una red mientras todavía se planea reutilizar el grafo de gradientes creado con esos mismos pesos. El enunciado efectúa la actualización **después** de recolectar el lote.

**REINFORCE es *on-policy*:** la pérdida debe construirse con acciones que se muestrearon según la política que se está actualizando. No reutilices arbitrariamente episodios antiguos tras muchas actualizaciones.

## 13. Reproducibilidad, almacenamiento y evaluación

El esqueleto establece **semillas**, configura `Adam`, registra métricas con TensorBoard y guarda una `policy.pt` con `state_dict`. Conviene saber interpretar su salida, pero **no hay que reimplementar estas partes**.

- `torch.manual_seed(seed)` controla fuentes aleatorias de PyTorch; NumPy y el entorno también pueden requerir semillas.
- Usar varias semillas (`--seeds 0-4`) ayuda a observar la variabilidad de un algoritmo estocástico.
- `policy.state_dict()` almacena los **parámetros** de la red, no su definición en Python.
- La métrica fundamental de la entrega es el **número de manzanas por partida**, que no equivale necesariamente al retorno de recompensas, sobre todo si se modifica el diseño de recompensa.

In [ ]:
# Guardado y carga de una red de ejemplo, sin tocar los ficheros de la entrega.
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta_tmp:
    ruta_pesos = Path(carpeta_tmp) / "pesos_demostracion.pt"
    torch.save(red.state_dict(), ruta_pesos)
    red_recuperada = RedDidactica(4, 2)
    try:
        pesos = torch.load(ruta_pesos, map_location="cpu", weights_only=True)
    except TypeError:  # Compatibilidad con instalaciones antiguas de PyTorch
        pesos = torch.load(ruta_pesos, map_location="cpu")
    red_recuperada.load_state_dict(pesos)
    assert torch.allclose(red(torch.ones(4)), red_recuperada(torch.ones(4)))
    print("Pesos guardados y restaurados correctamente.")

## 14. TensorBoard: registrar y leer métricas

El laboratorio guarda información mediante `torch.utils.tensorboard.SummaryWriter`. En particular, el script original utiliza `rollout/ep_return`, `rollout/ep_score`, `train/policy_loss`, `train/value_loss`, `eval/mean_reward` y `eval/mean_score`.

Los gráficos de entrenamiento no prueban por sí solos que la política haya aprendido: la función de pérdida de REINFORCE **no** tiene por qué disminuir continuamente. Compara sobre todo **el desempeño en evaluación**, usando el mismo protocolo entre variantes.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tb_tmp:
    writer_demo = SummaryWriter(log_dir=str(Path(tb_tmp) / "ejemplo"))
    for paso, puntuacion in [(0, 0.0), (100, 0.2), (200, 0.5)]:
        writer_demo.add_scalar("eval/mean_score_ejemplo", puntuacion, paso)
    writer_demo.flush()
    writer_demo.close()
    print("Ejemplo de eventos de TensorBoard generado correctamente.")

print("En un terminal, dentro del proyecto: tensorboard --logdir <directorio-de-eventos>")

La entrega utiliza un directorio determinado por `tb_dir(LAB)` en `common3.py`. **No adivines su ruta**: consulta esa función o busca el directorio de eventos que crea el entrenamiento.

Para las gráficas agregadas según el formato del repositorio, el fichero proporcionado recomienda:

```bash
python experiments_3/plot_results.py logs/lab_3_3 --metric score
```

La referencia comparativa `lab_3_3_referencia_sb3.py` utiliza otra implementación preparada por el curso. **No se permite** usar bibliotecas con una implementación ya hecha de REINFORCE como sustituto de los `TODO`.

## 15. Diseño de observaciones y recompensas para Snake (`TODO 6` y `TODO 7`)

**No disponemos en este cuaderno del código de `snake_design.py`.** Por tanto, estas son **orientaciones generales**, no una especificación de sus clases ni de los nombres que debéis utilizar.

### Observación (`SnakeObservation`)

Debe ser un vector numérico que permita a la política decidir. Para diseñarlo, pregunta:

- ¿Detecta peligros inmediatos (pared, cuerpo) con la información disponible?
- ¿Representa de forma útil la orientación del movimiento y la posición relativa de la manzana?
- ¿Cambia la longitud del vector según el estado? **No debería**: la red necesita una dimensión fija.
- ¿Todos los componentes tienen escala coherente y contienen valores finitos (`NaN` / `Inf` prohibidos)?
- ¿La información depende de datos accesibles para la política y no de resultados futuros?

### Recompensa (`SnakeReward`), opcional y valorada

La recompensa debe orientar el comportamiento sin crear **atajos indeseados**: por ejemplo, dar una recompensa positiva simplemente por acercarse a la manzana podría motivar conductas arriesgadas. Hay que considerar por separado comer, sobrevivir, chocar y dilatar la partida. **Conserva `score` como la métrica de comparación** entre diseños de recompensa.

**Antes de programar**, abre `snake_design.py`, lee las interfaces reales de `SnakeObservation`, `SnakeReward`, `make_snake_env` y `check_design`, y anota qué parámetros y formatos exige el repositorio.

## 16. `terminated` frente a `truncated`: cuestión para el informe

En la entrega se pregunta por la diferencia entre terminar por **choque** y por **demasiados pasos sin comer**. Según el texto suministrado, el primer caso se corresponde con `terminated` y el segundo con `truncated`.

- **REINFORCE episódico de la entrega:** en ambos casos termina la trayectoria y se calculan retornos desde las recompensas **observadas**. Si una trayectoria se trunca antes de su final natural, esos retornos no incluyen recompensas posteriores: pueden ser **retornos censurados**.
- **Baseline de la entrega:** aprende a aproximar esos retornos calculados. La truncación puede introducir un sesgo respecto al valor de la tarea sin límite artificial. En algoritmos con *bootstrap* se suele distinguir la truncación para decidir si añadir una estimación del valor final; el esqueleto proporcionado **no solicita** ese ajuste.

En vuestro informe, separad lo que **hace exactamente vuestra implementación** de lo que **podría hacerse** en otra variante del algoritmo.

## 17. Lanzar experimentos: `argparse`, opciones y visualización

Los experimentos se ejecutan en **terminal**, no como funciones del cuaderno. El fichero usa `argparse` para leer las opciones que se escriben después del nombre del script. Si no se indica una opción, se aplica su valor por defecto.

| Opción de la entrega | Qué controla | Valor predeterminado |
|---|---|---|
| `--steps` | Presupuesto total aproximado de pasos de entorno | `200000` |
| `--batch-episodes` | Partidas completas antes de cada actualización | `8` |
| `--gamma` | Factor de descuento de recompensas | `0.99` |
| `--hidden` | Unidades de cada capa oculta | `64` |
| `--lr`, `--lr-value` | Tasas de aprendizaje de la política y el valor | `0.001` |
| `--baseline` | Activa la red de valor | Desactivada |
| `--no-normalize` | Desactiva la normalización de retornos/ventajas | Normalización activada |
| `--eval-every` | Separación de evaluaciones en pasos | `25000` |
| `--eval-episodes` | Número de episodios por evaluación | `20` |
| `--obs` | Tipo de observación de Snake | `original` |
| `--reward` | Diseño de recompensa (`sparse`, `distance`, `custom`) | `sparse` |
| `--grid` | Tamaño del tablero de Snake | `12` |

**Ejemplo:** al indicar `--steps 50000` se sobrescribe *solo* ese argumento, no todos los demás.

El script permite reproducir la política guardada con `--watch RUTA/policy.pt`. En modo local puede abrir una ventana; con `--gif RUTA/salida.gif` emplea `render_mode='rgb_array'` y la biblioteca `imageio` para generar una animación. Para utilizar esa opción debe estar disponible `imageio` (incluida en la preparación). Esto **solo tiene sentido después** de entrenar y guardar la política, y requiere los componentes gráficos necesarios en el entorno de ejecución.

```bash
python experiments_3/lab_3_3_reinforce.py --watch logs/lab_3_3/<config>/<seed>/policy.pt
```

La ruta anterior es **ilustrativa**: utiliza la ruta real que imprima tu entrenamiento (`out`), no presupongas cómo organiza `run_dir(...)` los archivos.

In [ ]:
# Un ejemplo autocontenido de argparse: no modifica la entrega.
import argparse
p_ejemplo = argparse.ArgumentParser()
p_ejemplo.add_argument("--steps", type=int, default=200_000)
p_ejemplo.add_argument("--baseline", action="store_true")
config_ejemplo = p_ejemplo.parse_args(["--steps", "50000", "--baseline"])
print("Pasos:", config_ejemplo.steps, "| Baseline:", config_ejemplo.baseline)
assert config_ejemplo.steps == 50_000 and config_ejemplo.baseline is True

## 18. Diagnóstico rápido: síntomas y causas probables

| Síntoma | Comprobar primero |
|---|---|
| `RuntimeError: mat1 and mat2 shapes cannot be multiplied` | `obs_dim`, lote, dimensiones de `Linear` |
| `Categorical` recibe probabilidades incorrectas | ¿Se están pasando logits a `logits=` sin aplicar `softmax` antes? |
| Error de tipos al calcular `log_prob` | Índices de acción enteros (`torch.long`), no vectores one-hot |
| `loss.backward()` falla o los gradientes son `None` | ¿Los `log_probs` se han convertido a Python o se les ha aplicado `detach`? |
| `loss_pi` devuelve vector, no escalar | ¿Se ha agregado (`mean`) sobre **todos los pasos**? |
| La red de valor recibe gradiente del actor | ¿Se ha separado el baseline al calcular la ventaja? |
| `NaN` en pérdidas | Recompensas/observaciones finitas, normalización y escala de retornos |
| Aprendizaje muy inestable | Varianza entre semillas, tasa de aprendizaje, normalización, baseline |
| Recompensa sube y `score` no | Posible desalineación entre recompensa diseñada y manzanas |
| Buen resultado al entrenar, malo al evaluar | Diferencia entre muestreo de acciones y `argmax`; sobreajuste o alta varianza |

**Autocomprobaciones oficiales del ejercicio:**

```bash
python experiments_3/lab_3_3_reinforce.py --check
```

Las pruebas del esqueleto comprueban formas, tipos, gradientes, signo de la pérdida y partes del diseño; **superarlas no garantiza que el agente aprenda**.

## 19. Puente directo a cada `TODO` de la entrega

| Entrega | Has practicado en | Lo que te corresponde implementar |
|---|---|---|
| **TODO 1. `PolicyNet`** | Secciones 2–4 | MLP con **dos** ocultas `tanh`, logits `[B,n_actions]` |
| **TODO 2. `reward_to_go`** | Sección 6 | Recurrencia hacia atrás para cualquier lista de recompensas |
| **TODO 3. `collect_episode`** | Secciones 4, 5, 7 y 8 | Muestrear política y devolver el diccionario contractual con gradientes |
| **TODO 4. `policy_loss`** | Secciones 9 y 10 | Concatenar retornos de episodios, normalizar si procede, media del producto negativo |
| **TODO 5. `ValueNet + baseline`** | Sección 11 | Valor escalar por estado, ventaja, `detach` y error cuadrático medio |
| **TODO 6. `SnakeObservation`** | Sección 15 | Diseñar e implementar según `snake_design.py` |
| **TODO 7. `SnakeReward`** | Sección 15 | Diseño opcional de recompensa, justificando sus efectos |

El propio esqueleto prescribe que se implementen **en ese orden**, que se ejecuten las autocomprobaciones y que se compare la versión vanilla, la versión con baseline y el algoritmo de referencia.

## 20. Laboratorio guiado: orden de ejecución sugerido

**Fase A — Sin entrenar:** implementa `PolicyNet` y `reward_to_go`; ejecuta `--check`. Implementa `collect_episode`, verifica las formas del diccionario y el gradiente de `log_probs`; vuelve a ejecutar `--check`.

**Fase B — REINFORCE vanilla:** implementa `policy_loss`, comprueba el signo con `--check` y comienza por una sola semilla. Si no aprende, compara primero con `CartPole-v1` antes de cambiar el diseño de Snake.

**Fase C — Reducción de varianza:** implementa `ValueNet`, añade el baseline sin fuga de gradientes y prueba varias semillas. Explica la normalización utilizada.

**Fase D — Diseño y comparación:** implementa la observación propia, evalúa la recompensa opcional, registra `score` y contrasta las curvas entre configuraciones y semillas.

**Comandos propuestos por la entrega** (ejecutar desde la raíz del repositorio, no desde este cuaderno):

```bash
python experiments_3/lab_3_3_reinforce.py --check
python experiments_3/lab_3_3_reinforce.py --env CartPole-v1 --seeds 0 --steps 50000
python experiments_3/lab_3_3_reinforce.py --seeds 0
python experiments_3/lab_3_3_reinforce.py --seeds 0-4 --obs custom --reward sparse
python experiments_3/lab_3_3_reinforce.py --seeds 0-4 --obs custom --reward sparse --baseline
python experiments_3/lab_3_3_referencia_sb3.py --seeds 0-4 --obs custom --reward sparse
python experiments_3/plot_results.py logs/lab_3_3 --metric score
```

**No se garantiza que estos comandos funcionen fuera del repositorio**, porque dependen de los ficheros del curso y de sus implementaciones específicas.

## 21. Autoevaluación final

Antes de abrir el fichero de entrega, comprueba que puedes responder **sin consultar** las soluciones:

1. ¿Cuál es la diferencia entre `[obs_dim]` y `[B, obs_dim]`?
2. ¿Por qué la política devuelve logits en vez de probabilidades?
3. ¿Qué hace `Categorical(...).sample()` y qué tipo tiene la acción?
4. ¿Por qué el campo `log_probs` de un episodio debe conservar `requires_grad=True`?
5. ¿Por qué el cálculo de $G_t$ se puede hacer desde el final de la lista?
6. ¿Cuál es la diferencia entre `terminated` y `truncated`?
7. ¿Por qué el objetivo de la política lleva un **signo menos**?
8. ¿Qué significa que el baseline no debe recibir gradiente de `loss_pi`?
9. ¿Por qué conviene comparar varias semillas y evaluar el **score**, no solo la pérdida?
10. ¿Por qué completar `--check` no demuestra que Snake haya aprendido?

**Criterio recomendado para pasar a la entrega:** haber ejecutado sin errores los ejemplos y poder justificar los puntos 1–10.

---

### Recursos técnicos de consulta

- [PyTorch: tensores, autograd y entrenamiento](https://pytorch.org/tutorials/beginner/basics/intro.html).
- [PyTorch: distribuciones categóricas](https://docs.pytorch.org/docs/stable/distributions.html#categorical).
- [PyTorch: funcionamiento de autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html).
- [Gymnasium: API de entornos](https://gymnasium.farama.org/api/env/).
- [PyTorch: integración con TensorBoard](https://docs.pytorch.org/docs/stable/tensorboard.html).

**Fuente principal del diseño didáctico:** esqueleto de la entrega «Laboratorio 3.3 · REINFORCE desde cero en el Snake», suministrado por el profesorado. Los ejemplos independientes de este cuaderno no constituyen una implementación sustitutiva.